# SmartKitchen AI — Model Training & Evaluation

**BTEC Unit 21: Task 3 — Building the AI Solution (Learning Aim C)**

This notebook demonstrates:
- 80/20 train-test split
- Linear Regression training (baseline)
- Random Forest Regressor training (main model)
- Model evaluation: MAE, RMSE, R²
- 5-fold cross-validation
- Feature importance analysis
- Model comparison and selection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
print('All libraries loaded!')

## 1. Load Cleaned Data

In [ ]:
df = pd.read_csv('../data/processed/smartkitchen_clean.csv')
print(f'Cleaned dataset: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

## 2. Define Features and Target

In [ ]:
# Target variable
TARGET = 'waste_kg'

# Feature columns (all except target)
FEATURES = [col for col in df.columns if col != TARGET]

X = df[FEATURES]
y = df[TARGET]

print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'\nTarget: {TARGET}')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'y mean: {y.mean():.3f} kg, std: {y.std():.3f} kg')

## 3. Train-Test Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Testing set:  {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\nTrain mean waste: {y_train.mean():.3f} kg')
print(f'Test mean waste:  {y_test.mean():.3f} kg')

## 4. Model 1: Linear Regression (Baseline)

In [ ]:
# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict
lr_pred = lr_model.predict(X_test)

# Evaluate
lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)

print('=== LINEAR REGRESSION (Baseline) ===')
print(f'  MAE:    {lr_mae:.4f} kg')
print(f'  RMSE:   {lr_rmse:.4f} kg')
print(f'  R²:     {lr_r2:.4f}')

# Cross-validation
lr_cv = cross_val_score(lr_model, X, y, cv=5, scoring='r2')
print(f'  CV R² (5-fold): {lr_cv.mean():.4f} ± {lr_cv.std():.4f}')

## 5. Model 2: Random Forest Regressor (Main Model)

In [ ]:
# Train Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,      # 100 decision trees
    max_depth=15,          # Maximum tree depth
    min_samples_split=5,   # Minimum samples to split
    min_samples_leaf=2,    # Minimum samples in leaf
    random_state=42,
    n_jobs=-1              # Use all CPU cores
)
rf_model.fit(X_train, y_train)

# Predict
rf_pred = rf_model.predict(X_test)

# Evaluate
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print('=== RANDOM FOREST REGRESSOR (Main) ===')
print(f'  MAE:    {rf_mae:.4f} kg')
print(f'  RMSE:   {rf_rmse:.4f} kg')
print(f'  R²:     {rf_r2:.4f}')

# Cross-validation
rf_cv = cross_val_score(rf_model, X, y, cv=5, scoring='r2')
print(f'  CV R² (5-fold): {rf_cv.mean():.4f} ± {rf_cv.std():.4f}')

## 6. Model Comparison

In [ ]:
# Comparison table
comparison = pd.DataFrame({
    'Metric': ['MAE (kg)', 'RMSE (kg)', 'R² Score', 'CV R² Mean', 'CV R² Std'],
    'Linear Regression': [f'{lr_mae:.4f}', f'{lr_rmse:.4f}', f'{lr_r2:.4f}', f'{lr_cv.mean():.4f}', f'{lr_cv.std():.4f}'],
    'Random Forest': [f'{rf_mae:.4f}', f'{rf_rmse:.4f}', f'{rf_r2:.4f}', f'{rf_cv.mean():.4f}', f'{rf_cv.std():.4f}']
})
print('\n=== MODEL COMPARISON ===')
print(comparison.to_string(index=False))

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

metrics = ['MAE', 'RMSE', 'R²']
lr_vals = [lr_mae, lr_rmse, lr_r2]
rf_vals = [rf_mae, rf_rmse, rf_r2]

for i, (metric, lr_v, rf_v) in enumerate(zip(metrics, lr_vals, rf_vals)):
    axes[i].bar(['Linear Reg.', 'Random Forest'], [lr_v, rf_v], color=['#3b82f6', '#10b981'])
    axes[i].set_title(f'{metric} Comparison')
    axes[i].set_ylabel(metric)

plt.tight_layout()
plt.savefig('../reports/figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Importance (Random Forest)

In [ ]:
# Feature importance
importances = pd.Series(rf_model.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
importances.plot(kind='barh', color='#10b981', ax=ax)
ax.set_xlabel('Importance Score')
ax.set_title('Random Forest Feature Importances')
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 Most Important Features:')
for feat, imp in importances.sort_values(ascending=False).head(5).items():
    print(f'  {feat}: {imp:.4f}')

## 8. Prediction vs Actual (Residual Analysis)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Predicted vs Actual
axes[0].scatter(y_test, rf_pred, alpha=0.5, s=20, color='#10b981')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Waste (kg)')
axes[0].set_ylabel('Predicted Waste (kg)')
axes[0].set_title('Predicted vs Actual (Random Forest)')

# Residuals
residuals = y_test - rf_pred
axes[1].hist(residuals, bins=30, color='#3b82f6', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.savefig('../reports/figures/prediction_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Trained Models

In [ ]:
# Save models
joblib.dump(lr_model, '../models/linear_regression.joblib')
joblib.dump(rf_model, '../models/random_forest.joblib')

print('✅ Models saved:')
print('   - models/linear_regression.joblib')
print('   - models/random_forest.joblib')
print(f'\n📊 Final recommendation: Random Forest (R²={rf_r2:.4f} vs Linear R²={lr_r2:.4f})')

## 10. Conclusion

### Model Selection Decision:

| Criteria | Linear Regression | Random Forest |
|----------|:-----------------:|:-------------:|
| Interpretability | ★★★★★ | ★★★ |
| Accuracy (R²) | Good | Better |
| Handles non-linearity | ❌ | ✅ |
| Feature interactions | ❌ | ✅ |
| Overfitting risk | Low | Moderate |

**Decision:** We use **Random Forest** as the production model because:
1. Higher R² score on test data
2. Can capture non-linear relationships (e.g., holiday + hot weather)
3. Built-in feature importance for explainability
4. Cross-validation confirms generalization ability

Linear Regression remains as a **baseline** for comparison.